# Chapter 12 Walkthrough: Object-Oriented Programming (OOP) for Data Scientists & Researchers

**Book Reference:** Chapter 12, Sections 12.1 – 12.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections.

## Section 12.3.1 — Classes, `__init__`, `self`, and Instance State

In [ ]:
class ClinicalSample:
    """Anthropometric record pairing patient data with BMI behavior."""

    def __init__(self, patient_id, weight_kg, height_cm):
        self.patient_id = patient_id
        self.weight_kg = weight_kg
        self.height_cm = height_cm

    def bmi(self):
        height_m = self.height_cm / 100
        return round(self.weight_kg / (height_m ** 2), 2)

    def __repr__(self):
        return (
            f"ClinicalSample(patient_id={self.patient_id!r}, "
            f"weight_kg={self.weight_kg}, height_cm={self.height_cm})"
        )

s1 = ClinicalSample("P-101", 64.5, 168.0)
s2 = ClinicalSample("P-102", 54.6, 156.0)

print(s1)
print("s1 BMI:", s1.bmi(), "| s2 BMI:", s2.bmi())
print("Independent objects?", s1 is not s2)
print("Is ClinicalSample?  ", isinstance(s1, ClinicalSample))


## Section 12.3.2 — Structured Scientific Records with `@dataclass`

In [ ]:
from dataclasses import dataclass

@dataclass
class VitalsSnapshot:
    """Single bedside vitals observation."""
    patient_id: str
    temp_c: float
    sbp_mmhg: int

    def has_fever(self):
        return self.temp_c >= 38.0

v1 = VitalsSnapshot("P-201", 38.4, 128)
v2 = VitalsSnapshot(patient_id="P-201", temp_c=38.4, sbp_mmhg=128)
v3 = VitalsSnapshot("P-202", 36.8, 118)

print("v1:", v1)
print("v1 has fever?: ", v1.has_fever())
print("v1 == v2 value:", v1 == v2, "| same object?:", v1 is v2)
print("v1 == v3 value:", v1 == v3)


## Section 12.3.3 — Dunder Protocols (`__len__`) & Object Composition

In [ ]:
class WardMonitor:
    """Composes VitalsSnapshot objects into an isolated ward tracker."""

    def __init__(self, ward_name):
        self.ward_name = ward_name
        self.records = []  # Always create mutable lists inside __init__!

    def add_record(self, snapshot):
        self.records.append(snapshot)

    def __len__(self):
        return len(self.records)

    def fever_count(self):
        return sum(1 for r in self.records if r.has_fever())

ward_a = WardMonitor("Dhaka Ward A")
ward_b = WardMonitor("Sylhet Ward B")

ward_a.add_record(v1)
ward_a.add_record(v3)

for w in (ward_a, ward_b):
    print(f"{w.ward_name}: len={len(w)}, fevers={w.fever_count()}")


## Section 12.3.4 — The ML Bridge: Mini Scikit-Learn (`fit`/`transform`) & Mini PyTorch (`forward`/`__call__`)

In [ ]:
class MiniStandardizer:
    """Scikit-Learn style transformer: fit on train, transform test."""

    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, values):
        mean = sum(values) / len(values)
        var = sum((x - mean) ** 2 for x in values) / len(values)
        self.mean_ = mean
        self.std_ = var ** 0.5
        return self

    def transform(self, values):
        if self.std_ == 0:
            return [0.0 for _ in values]
        return [round((x - self.mean_) / self.std_, 3) for x in values]

class RiskNeuron:
    """Mini PyTorch nn.Module pattern: __call__ delegates to forward."""

    def __init__(self, weight, bias):
        self.weight = weight
        self.bias = bias

    def forward(self, x):
        return round(self.weight * x + self.bias, 3)

    def __call__(self, x):
        return self.forward(x)

train_hb = [10.0, 11.0, 12.0, 13.0]
test_hb = [9.5, 11.5, 13.5]

scaler = MiniStandardizer().fit(train_hb)
z_test = scaler.transform(test_hb)
print("Fitted mean_:", scaler.mean_, "| z_test:", z_test)

neuron = RiskNeuron(weight=-0.8, bias=1.2)
print("forward(z):  ", [neuron.forward(z) for z in z_test])
print("neuron(z):   ", [neuron(z) for z in z_test])


## Section 12.3.5 — Break It On Purpose! (OOP Error Anatomy)

In [ ]:
# Intentional Bug #1: AttributeError — reading unfitted attribute
class RawMeanScaler:
    def fit(self, values):
        self.mean_ = sum(values) / len(values)
        return self

unfitted_scaler = RawMeanScaler()
print(unfitted_scaler.mean_)


In [ ]:
# Intentional Bug #2: TypeError — missing `self` in method definition
class Sample:
    def __init__(self, weight_kg):
        self.weight_kg = weight_kg

    def double():
        return self.weight_kg * 2

sample = Sample(60.0)
print(sample.double())


In [ ]:
# Intentional Bug #3: ValueError — domain guard rejecting bad vital sign
class GuardedPatient:
    def __init__(self, patient_id, hb_g_dl):
        if not (2.0 <= hb_g_dl <= 25.0):
            raise ValueError(
                f"Invalid hb_g_dl={hb_g_dl}: expected [2.0, 25.0]"
            )
        self.patient_id = patient_id
        self.hb_g_dl = hb_g_dl

bad_patient = GuardedPatient("P-999", 999.0)


## Section 12.4 — Worked Problem-Structuring Blueprint: Composable Clinical ML Pipeline

In [ ]:
class SentinelMeanImputer:
    """Learns clean mean on training data and fills None/999 safely."""

    def __init__(self):
        self.mean_ = None

    def fit(self, values):
        valid = [v for v in values if v is not None and v != 999]
        self.mean_ = round(sum(valid) / len(valid), 2)
        return self

    def transform(self, values):
        if self.mean_ is None:
            raise RuntimeError("Fit imputer before calling transform().")
        return [
            self.mean_ if (v is None or v == 999) else v
            for v in values
        ]

class ClinicalRiskPipeline:
    """Composes an imputer, standardizer, and callable linear scorer."""

    def __init__(self, weight=-1.5, bias=0.5):
        self.imputer = SentinelMeanImputer()
        self.scaler = MiniStandardizer()
        self.scorer = RiskNeuron(weight=weight, bias=bias)

    def fit(self, raw_train):
        clean_train = self.imputer.fit(raw_train).transform(raw_train)
        self.scaler.fit(clean_train)
        return self

    def predict_scores(self, raw_values):
        imputed = self.imputer.transform(raw_values)
        scaled = self.scaler.transform(imputed)
        return [self.scorer(z) for z in scaled]

raw_train = [10.4, None, 11.8, 999, 13.2]
raw_test = [9.8, None, 12.5]

pipe = ClinicalRiskPipeline(weight=-1.5, bias=0.5).fit(raw_train)
scores = pipe.predict_scores(raw_test)

print("Train mean_ learned:", pipe.imputer.mean_)
print("Test risk scores:   ", scores)
print("Raw test untouched: ", raw_test)
